In [ ]:
pip install datasets

In [ ]:

from datasets import load_dataset

# Stream the dataset and take the first 25k rows
ds_stream = load_dataset("nvidia/Nemotron-Personas-India", split='en_IN', streaming=True)
first_25k = ds_stream.take(25000)

# Convert to a standard list or pandas DataFrame if needed
import pandas as pd
df = pd.DataFrame(list(first_25k))

In [ ]:
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    elif hasattr(obj, "item"):  # numpy types
        return obj.item()
    else:
        return obj

In [ ]:
import requests
import json

url = "https://api.longcat.chat/openai/v1/chat/completions"

headers = {
    "Authorization": "Bearer your_api_key_here",
    "Content-Type": "application/json"
}

def call_llm(prompt):
    data = {
        "model": "LongCat-Flash-Lite",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1000,
        "temperature": 0.85
    }

    response = requests.post(url, headers=headers, json=data)
    

    return response.json()

In [ ]:
def augment_text(text, df):

    # persona = make_json_safe(sample_persona(df))

   

    

    try:
        output = response["choices"][0]["message"]["content"]
        variations = json.loads(output)
    except:
        variations = []

    return variations

In [ ]:
import pandas as pd

original_df = pd.read_csv("C:\\Users\\user\\Downloads\\140924_combined_reports_prescriptions_7600.csv")

In [ ]:
!pip install tqdm rich

In [ ]:
def augment_text(text, df):

    # persona = make_json_safe(sample_persona(df))

    try:
        output = response["choices"][0]["message"]["content"]
        variations = json.loads(output)
    except:
        variations = []

    return variations

In [ ]:
PROMPT_TEMPLATE_MEDICAL_OTHERS = """
You are a data augmentation assistant.

Given an input text that may look like a medical report, lab result, or prescription, your task is to transform it into formats that DO NOT resemble structured medical documents.

Your goal is to generate outputs that belong to the "other" class in a classification dataset.

### Instructions:
1. Convert the input into ONE of the following styles (randomly choose):
   - Casual conversation (2–3 people talking)
   - WhatsApp/chat messages
   - Story or narrative paragraph
   - General discussion or explanation
   - Blog-style writing
   - Question-answer dialogue
   - Personal note or diary entry

2. IMPORTANT:
   - Do NOT preserve structured medical formatting (no bullet lists like prescriptions, no "Name:", "Age:", etc.)
   - Do NOT keep it looking like a report
   - You MAY keep or slightly distort the meaning, but presentation must change completely
   - You MAY omit, reorder, or generalize details
   - You MAY introduce informal language, filler words, or conversational tone


3. Optional Noise Injection:
   - Add small talk, interruptions, or irrelevant details
   - Introduce ambiguity or partial information
   - Slightly alter names, numbers, or entities

4. - Do NOT preserve structured medical formatting  
  (no "Name:", "Age:", bullet lists, tables, or prescription layout)
- Break ordering of information (shuffle, drop, merge details)
- Use natural, human-like phrasing
- You MAY distort or generalize the meaning

5.  OCR Noise + Typo Injection (VERY IMPORTANT):

Introduce realistic OCR errors and human typing mistakes:

- Character substitutions:
  - o → 0, l → 1, e → c, a → @, s → 5
- Missing or extra spaces:
  - "bloodpressure" / "blood  pressure"
- Broken or merged words:
  - "medicationtaken" / "medi cation"
- Random capitalization:
  - "pAtient", "DOcTor"
- Spelling mistakes:
  - "fever" → "fevr", "tablet" → "tablct"
- Punctuation noise:
  - extra commas, missing full stops, "??", "..."
- Slight numeric distortions:
  - 500 → 50O, 25 → 2S
- Partial truncation:
  - cut words midway ("prescrip...", "medic...")

 Do NOT overdo noise — keep text readable but imperfect (like OCR output).

6. Output must:
   - Be natural and human-like
   - Clearly NOT resemble a medical document
   - Be suitable for classification as "other"

### Input:
{text}

OUTPUT FORMAT:


Return ONLY a JSON array of string:

[
"",

]


"""

In [ ]:
import json
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time

# --- CONSTANTS ---
INPUT_CSV = "C:\\Users\\user\\Downloads\\140924_combined_reports_prescriptions_7600.csv"       # Your extracted text from the Aadhar OCR
OUTPUT_JSONL = "augmented_data.jsonl" # This will save results line-by-line
MAX_WORKERS = 5                     # Don't go too high or the API will hang
TIMEOUT = 45                        # Seconds to wait for LLM before giving up

# Thread lock to prevent multiple threads from writing to the file at the same time
file_lock = threading.Lock()

def save_result(data_list):
    """Safely appends results to the JSONL file."""
    with file_lock:
        with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
            for entry in data_list:
                f.write(json.dumps(entry) + "\n")

def get_processed_files():
    """Checks the output file to see which filenames are already finished."""
    if not os.path.exists(OUTPUT_JSONL):
        return set()
    
    processed = set()
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                processed.add(data["filename"])
            except:
                continue
    return processed

def process_row(idx, row):
    """Your logic for one row."""
    try:
        text = row["text"]
        label = row.get("label", "N/A")
        filename = row.get("filename", f"row_{idx}")

        if pd.isna(text) or text.strip() == "":
            return None

        # Build your prompt
        prompt_other = PROMPT_TEMPLATE_MEDICAL_OTHERS.format(text=text)

        # Call LLM (Make sure your call_llm function has an internal timeout!)
        # Example: response = client.chat.completions.create(..., timeout=30)
        response = call_llm(prompt_other) 
        
        output = response["choices"][0]["message"]["content"]
        variations = json.loads(output)

        local_rows = []
        # Store Original
        local_rows.append({
            "filename": filename,
            "original_text": text,
            "augmented_text": text,
            "label": label,
            "status": "original"
        })
        # Store Augmented
        for v in variations:
            local_rows.append({
                "filename": filename,
                "original_text": text,
                "augmented_text": v,
                "label": label,
                "status": "augmented"
            })
        return local_rows

    except Exception as e:
        # Logging errors to console so you see why it failed
        print(f"\n[ERROR] Row {idx} ({filename}): {str(e)}")
        return None

def main():
    # 1. Load Data
    df = pd.read_csv(INPUT_CSV)
    
    # 2. Check for Checkpoints (Resume Logic)
    processed_filenames = get_processed_files()
    print(f"Total rows: {len(df)}")
    print(f"Already processed: {len(processed_filenames)}")
    
    # Filter out rows already in the JSONL
    pending_df = df[~df['filename'].isin(processed_filenames)]
    print(f"Rows remaining: {len(pending_df)}")

    if pending_df.empty:
        print("Everything is already processed!")
        return

    # 3. Start Multi-threaded Processing
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Map futures to filenames
        future_to_row = {
            executor.submit(process_row, idx, row): row['filename'] 
            for idx, row in pending_df.iterrows()
        }

        # tqdm for progress bar
        for future in tqdm(as_completed(future_to_row), total=len(future_to_row), desc="Processing"):
            filename = future_to_row[future]
            try:
                # We add a timeout here. If the thread hangs for > 60s, it raises TimeoutError
                result = future.result(timeout=TIMEOUT)
                if result:
                    # 4. SAVE SIMULTANEOUSLY TO DISK
                    save_result(result)
            except Exception as e:
                print(f"\n[CRITICAL] {filename} timed out or failed: {e}")

    print(f"\nDone! Results saved to {OUTPUT_JSONL}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

aug_df = pd.DataFrame(augmented_rows)

In [ ]:
aug_df.to_csv("C:\\Users\\user\\Documents\\augmented_dataset_conversation_other.csv", index=False)